# Reproducibility and October-Readiness Evidence | Issue #14

Perform a clean reproducibility check of the September workflow and record the exact project, source, runtime, and artifact versions used for Gate 3.

This notebook:

- verifies the run begins from a clean committed project revision;
- reruns the final September notebooks;
- compares regenerated evidence against the frozen artifacts;
- records source, code, dependency, runtime, threshold, binning, and plot settings;
- creates Workflow Run and Artifact Bundle IDs;
- drafts the October evaluation plan;
- records final approval, disagreements, and limitations.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
import os
import platform
import re
import shutil
import subprocess
import uuid

import numpy as np
import pandas as pd


def find_repo_root():
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (path / ".git").exists():
            return path
    raise FileNotFoundError("Repository root not found.")


ROOT = find_repo_root()

SEPTEMBER = (
    ROOT
    / "notebooks"
    / "final-deliverables"
    / "September"
)

GATE1 = SEPTEMBER / "Gate 1"
GATE2 = SEPTEMBER / "Gate 2"
GATE3 = SEPTEMBER / "Gate 3"

OUTPUT_DIR = (
    GATE3
    / "reproducibility-and-october-readiness"
)

RERUN_DIR = OUTPUT_DIR / "rerun_notebooks"
SNAPSHOT_DIR = OUTPUT_DIR / "baseline_snapshot"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RERUN_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_PATH = ROOT / "data" / "allstate_claims_data.csv"

ROUNDING_DECIMALS = 6
NUMERIC_ATOL = 10 ** (-ROUNDING_DECIMALS)

RARE_LEVEL_THRESHOLD = 188
REQUESTED_CONTINUOUS_BINS = 10
TARGET_HISTOGRAM_BINS = 100

print("Repository:", ROOT)
print("Output:", OUTPUT_DIR.relative_to(ROOT))

Repository: C:\Users\stapl\BTT\Fall AI Studio\Allstate 1A\Allstate-1A-predicting-auto-claims-severity
Output: notebooks\final-deliverables\September\Gate 3\reproducibility-and-october-readiness


## 1. Record Clean Project and Source Identity

The reproducibility run must begin from a committed project revision.

Record:

- Git branch and commit;
- whether the working tree was clean at run start;
- source-data SHA-256;
- source file size;
- Python and notebook/runtime versions; and
- Workflow Run ID.

In [2]:
def git(*args):
    return subprocess.check_output(
        ["git", *args],
        cwd=ROOT,
        text=True
    ).strip()


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


GIT_COMMIT = git("rev-parse", "HEAD")
GIT_BRANCH = git("rev-parse", "--abbrev-ref", "HEAD")
GIT_STATUS_AT_START = git("status", "--porcelain")
STARTED_CLEAN = GIT_STATUS_AT_START == ""

if not STARTED_CLEAN:
    raise RuntimeError(
        "Final reproducibility run must begin from a clean committed "
        "working tree. Commit/stash changes and rerun."
    )

SOURCE_SHA256 = sha256_file(SOURCE_PATH)
SOURCE_SIZE = SOURCE_PATH.stat().st_size

UTC_START = datetime.now(timezone.utc)

WORKFLOW_RUN_ID = (
    f"G3-{UTC_START.strftime('%Y%m%dT%H%M%SZ')}-"
    f"{GIT_COMMIT[:8]}"
)

try:
    import google.colab  # noqa
    RUNTIME_CONTEXT = "Google Colab"
except ImportError:
    RUNTIME_CONTEXT = "VS Code / local Jupyter"

run_identity = {
    "workflow_run_id": WORKFLOW_RUN_ID,
    "run_started_utc": UTC_START.isoformat(),
    "git_branch": GIT_BRANCH,
    "git_commit": GIT_COMMIT,
    "working_tree_clean_at_start": STARTED_CLEAN,
    "source_path": SOURCE_PATH.relative_to(ROOT).as_posix(),
    "source_sha256": SOURCE_SHA256,
    "source_size_bytes": SOURCE_SIZE,
    "runtime_context": RUNTIME_CONTEXT,
    "python_version": platform.python_version(),
    "platform": platform.platform(),
}

display(pd.DataFrame([run_identity]))

RuntimeError: Final reproducibility run must begin from a clean committed working tree. Commit/stash changes and rerun.